In [1]:
pip install pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [6]:
from datetime import datetime


def preprocess_location(filepath, datetime_col='Datetime'):
    # Load data
    df = pd.read_csv(filepath, dayfirst=True)
    print(f"{filepath}: Raw rows = {len(df)}")
    
    # Convert datetime column to pandas datetime
    df[datetime_col] = pd.to_datetime(df[datetime_col], dayfirst=True, errors='coerce')
    df = df.dropna(subset=[datetime_col])
    
    # Sort by datetime
    df = df.sort_values(datetime_col)
    
    # Check NaNs before rolling
    print("NaNs before rolling:")
    print(df[['Outdoor_Temp', 'Indoor_Temp', 'Outdoor_Humidity', 'Indoor_Humidity']].isna().sum())
    
    # Apply rolling mean (24-hour window)
    rolled = df[['Outdoor_Temp', 'Indoor_Temp', 'Outdoor_Humidity', 'Indoor_Humidity']].rolling(window=24).mean()
    
    rolled = rolled.bfill()
    
    # Assign rolled data back to df
    df[['Outdoor_Temp', 'Indoor_Temp', 'Outdoor_Humidity', 'Indoor_Humidity']] = rolled.astype('float64')

    
    # Check NaNs after rolling
    print("NaNs after rolling:")
    print(df[['Outdoor_Temp', 'Indoor_Temp', 'Outdoor_Humidity', 'Indoor_Humidity']].isna().sum())
    
    # Fill NaNs caused by rolling at start (backward fill or forward fill)
    df.bfill(inplace=True)
  # or use method='ffill'
    
    # Final NaN check before dropping
    print("NaNs before final dropna:")
    print(df.isna().sum())
    
    # Drop any remaining NaNs
    df = df.dropna()
    
    # Add Year and Date columns
    df['Year'] = df[datetime_col].dt.year
    df['Date'] = df[datetime_col].dt.date
    
    # Split train/test datasets
    df_train = df[df['Year'] == 2025].copy()
    df_test = df[(df[datetime_col] >= "2023-11-14") & (df[datetime_col] <= "2024-07-10")].copy()
    
    if 'Bath_real' in filepath:
        train_df = pd.DataFrame()  # Empty
        test_df = df  # All Bath data used for testing
    else:
        train_df = df  # All synthetic data for training
        test_df = pd.DataFrame()  # No test data from synthetic sets
    
    print(f"{filepath}: Processed rows = {len(df)} → Train: {len(df_train)}, Test: {len(df_test)}\n")
    
    return df_train, df_test








In [7]:
# Load and preprocess synthetic training datasets (2025 only)
train_gatwick, _ = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv")
train_oslo, _ = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv")

# Load real testing dataset (Nov 2023 – July 2024)
_, test_bath = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Bath_real.csv")  # Update path if different


C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv: Raw rows = 8760
NaNs before rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs after rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs before final dropna:
Datetime            0
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv: Processed rows = 3456 → Train: 3456, Test: 0

C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv: Raw rows = 8760
NaNs before rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs after rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs before final dropna:
Datetime            0
Outdoor_Temp        0
Indoor_Temp         0
O

In [8]:
files = [
    r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv",
    r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv",

    r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Bath_real.csv"
]

for f in files:
    df = pd.read_csv(f)
    print(f"{f}: raw rows = {len(df)}")


C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv: raw rows = 8760
C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv: raw rows = 8760
C:\Users\Riya\Documents\heritage-predictor\env\Data\Bath_real.csv: raw rows = 5320


In [9]:
# 1. Load and preprocess datasets
train_gatwick, _ = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv")
train_norway, _ = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv")
_, test_bath = preprocess_location(r"C:\Users\Riya\Documents\heritage-predictor\env\Data\Bath_real.csv")

# 2. Combine all synthetic training datasets
train_combined = pd.concat([
    train_gatwick, train_norway
], ignore_index=True)

print(f"Combined training dataset shape: {train_combined.shape}")
print(f"Bath test dataset shape: {test_bath.shape}")

# 3. Optional: Split training data into train + validation sets (to tune and check your model)
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(train_combined, test_size=0.2, random_state=42)
print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")

# test_bath will be your final test set to evaluate your model's real-world performance


C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv: Raw rows = 8760
NaNs before rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs after rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs before final dropna:
Datetime            0
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
C:\Users\Riya\Documents\heritage-predictor\env\Data\Gatwick.csv: Processed rows = 3456 → Train: 3456, Test: 0

C:\Users\Riya\Documents\heritage-predictor\env\Data\Norway.csv: Raw rows = 8760
NaNs before rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs after rolling:
Outdoor_Temp        0
Indoor_Temp         0
Outdoor_Humidity    0
Indoor_Humidity     0
dtype: int64
NaNs before final dropna:
Datetime            0
Outdoor_Temp        0
Indoor_Temp         0
O

In [10]:
from sklearn.ensemble import RandomForestRegressor


# Define features
feature_cols = ['Outdoor_Temp', 'Outdoor_Humidity']

# Separate targets
y_train_temp = train_data['Indoor_Temp']
y_train_humidity = train_data['Indoor_Humidity']

y_val_temp = val_data['Indoor_Temp']
y_val_humidity = val_data['Indoor_Humidity']

y_test_temp = test_bath['Indoor_Temp']
y_test_humidity = test_bath['Indoor_Humidity']

# Train model for temperature
rf_temp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_temp.fit(train_data[feature_cols], y_train_temp)

# Train model for humidity
rf_hum = RandomForestRegressor(n_estimators=100, random_state=42)
rf_hum.fit(train_data[feature_cols], y_train_humidity)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
from sklearn.metrics import mean_squared_error, r2_score

# Predict and evaluate temperature
val_preds_temp = rf_temp.predict(val_data[feature_cols])
test_preds_temp = rf_temp.predict(test_bath[feature_cols])

print("Indoor Temperature:")
print(f"Validation MSE: {mean_squared_error(y_val_temp, val_preds_temp):.3f}, R2: {r2_score(y_val_temp, val_preds_temp):.3f}")
print(f"Test MSE: {mean_squared_error(y_test_temp, test_preds_temp):.3f}, R2: {r2_score(y_test_temp, test_preds_temp):.3f}")

# Predict and evaluate humidity
val_preds_hum = rf_hum.predict(val_data[feature_cols])
test_preds_hum = rf_hum.predict(test_bath[feature_cols])

print("\nIndoor Humidity:")
print(f"Validation MSE: {mean_squared_error(y_val_humidity, val_preds_hum):.3f}, R2: {r2_score(y_val_humidity, val_preds_hum):.3f}")
print(f"Test MSE: {mean_squared_error(y_test_humidity, test_preds_hum):.3f}, R2: {r2_score(y_test_humidity, test_preds_hum):.3f}")


Indoor Temperature:
Validation MSE: 3.697, R2: 0.939
Test MSE: 17.630, R2: -2.962

Indoor Humidity:
Validation MSE: 54.322, R2: 0.766
Test MSE: 113.140, R2: -3.502
